In [0]:
%pip install Pdfplumber langchain 
dbutils.library.restartPython()

In [0]:
import os
from pyspark.sql.functions import substring_index

directory_path = "/Volumes/workspace/rag/files_vol"

#lista os arquivos na pasta
file_paths = [file.path for file in dbutils.fs.ls(directory_path)]

#extrai os nomes dos arquivos pelo caminho
df = spark.createDataFrame(file_paths, "string").select(substring_index("value", "/", -1).alias("file_name"))

df.show()

In [0]:
%pip install langchain-text-splitters

import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_volume_path = directory_path

#lista de arquivos ja processados
processed_files = spark.sql(f"SELECT DISTINCT file_name FROM workspace.rag.docs_track").collect()
processed_files = set(row["file_name"] for row in processed_files)

#processar apenas documentos novos
new_files = [file for file in os.listdir(pdf_volume_path) if file not in processed_files]

all_text = ''  

for file_name in new_files:
    #extrair texto do pdf novo
    pdf_path = os.path.join(pdf_volume_path, file_name)

    with pdfplumber.open(pdf_path) as pdf:
        for pdf_page in pdf.pages:
            single_page_text = pdf_page.extract_text()
            # Separate each page's text with newline
            all_text = all_text + '\n' + single_page_text

# Split the combined text into chunks using the RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

length_function = len

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=200,
    length_function=length_function,
)
chunks = splitter.split_text(all_text)

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, StringType
import pandas as pd

@pandas_udf("array<string>")
def get_chunks(dummy):
    return pd.Series([chunks])

spark.udf.register("get_chunks_udf", get_chunks)

spark.sql("""
insert into workspace.rag.docs_text (text)
select explode(get_chunks_udf('dummy')) as text
""")

In [0]:
df.createOrReplaceTempView("temp_table")  

#insercao das novas linhas
spark.sql("""
    INSERT INTO workspace.rag.docs_track
    SELECT * FROM temp_table
    WHERE NOT EXISTS (
        SELECT 1 FROM workspace.rag.docs_track
        WHERE temp_table.file_name = workspace.rag.docs_track.file_name
    )
""")